In [1]:
import pandas as pd
import os
import numpy as np
import re

In [2]:
pd.set_option('display.max_columns', None)

# <span style="color:blue;">**Control**</span>

# Data preparation

## A. DICOM

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [4]:
file_path = "../Data/dicom_tag.xlsx"
dicom = pd.read_excel(file_path)

In [5]:
dicom["PatientID"].unique().size

5515

In [6]:
PIDs = dicom["PatientID"].unique()

#### Create a list of unique DICOM entries to serve as the reference linking the EHR to the images available for specific patient visits (as images are limited to certain visits, not all).

#### See shared parameters in [parameter exlanation][peid] to link data

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 

**shared parameters:**

DICOM ↔︎ EHR (pathology)

PatientID ↔︎ PATIENT_STUDY_ID 

AccessionNumber ↔︎ ACCESSION_NUMBER 

PatientBirthDate ↔︎ BIRTH_DATE

In [7]:
dicom.rename(columns={'PatientID': 'PATIENT_STUDY_ID', 'AccessionNumber': 'ACCESSION_NUMBER', 'PatientBirthDate': 'BIRTH_DATE'}, inplace=True)

In [8]:
dicom

,PATIENT_STUDY_ID,ACCESSION_NUMBER,BIRTH_DATE,PatientAge,PatientSex,StudyDate,StudyTime,AcquisitionDate,Modality,Manufacturer,ManufacturerModelName,StudyDescription,SeriesNumber,SeriesDescription,Exposure,Rows,Columns,PixelSpacing,study,side,series
0,4330018595,60103700,1965-07-01,054Y,F,2020-06-02,09:10:52,NaN,MG,"R2 Technology, Inc.",Cenova,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,1.0,Hologic R2 ImageChecker CAD SC,NaN,1500.0,1250.0,NaN,DIAG,NaN,NaN
1,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"R2 Technology, Inc.",Cenova,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,1.0,Hologic R2 ImageChecker CAD SC,NaN,1500.0,1250.0,NaN,DIAG,NaN,NaN
2,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,2020-06-02,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71100000.0,R ML,126.0,3328.0,2560.0,0.038889\0.038889,DIAG,R,NaN
3,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71300000.0,R XCCL Intelligent 2D,63.0,3328.0,2560.0,0.064318\0.064318,DIAG,R,IN2D
4,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71300000.0,R ML Intelligent 2D,57.0,3328.0,2560.0,0.064619\0.064619,DIAG,R,IN2D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136228,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,02:45:37,2019-10-28,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,72100000.0,L CC Tomosynthesis Projection,72.0,425.0,266.0,NaN,SCREEN,L,DBT
136229,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,02:45:37,2019-10-28,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,72100000.0,R CC Tomosynthesis Projection,74.0,425.0,266.0,NaN,SCREEN,R,DBT
136230,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,02:45:37,2019-10-28,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,72100000.0,L MLO Tomosynthesis Projection,78.0,425.0,266.0,NaN,SCREEN,L,DBT
136231,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,18:20:08,NaN,PR,Philips Medical Systems,iSite Enterprise,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,1.0,5DBAD14E0,NaN,NaN,NaN,NaN,SCREEN,NaN,NaN


In [9]:
# Define the key identifier columns and columns to extract
key_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'BIRTH_DATE', 'PatientSex']
extract_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'StudyTime', 'study']
unique_dicom = dicom.drop_duplicates(subset=key_columns, keep='first')
unique_dicom = unique_dicom[extract_columns]
unique_dicom.reset_index(drop=True, inplace=True)

In [10]:
unique_dicom['image_available'] = 1

In [11]:
unique_dicom

,PATIENT_STUDY_ID,ACCESSION_NUMBER,StudyDate,StudyTime,study,image_available
0,4330018595,60103700,2020-06-02,09:10:52,DIAG,1
1,4330018595,60690108,2020-06-02,08:33:36,SCREEN,1
2,4330018595,63737104,2019-08-19,02:11:20,DIAG,1
3,4330029102,64888584,2019-05-16,08:32:09,SCREEN,1
4,4330044371,61647696,2020-02-06,15:37:19,SCREEN,1
...,...,...,...,...,...,...
15135,4339943307,63434181,2019-06-17,17:07:31,DIAG,1
15136,4339943307,63570279,2019-05-23,18:36:05,SCREEN,1
15137,4339945522,62906727,2019-10-18,03:00:54,DIAG,1
15138,4339945522,450264459,2022-10-14,11:54:03,NaN,1


#### Add "dbt_available" column, if available = 1

In [12]:
dicom_DBT = dicom[dicom["series"]=="DBT"]
key_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'BIRTH_DATE', 'PatientSex']
extract_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'study', 'series']
unique_dicom_DBT = dicom_DBT.drop_duplicates(subset=key_columns, keep='first')[extract_columns]

In [13]:
unique_dicom = unique_dicom.merge(
    unique_dicom_DBT, 
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'study'],
    how='left')

In [14]:
unique_dicom = unique_dicom.rename(columns={'series': 'dbt_available'})
unique_dicom['dbt_available'] = unique_dicom['dbt_available'].replace('DBT', 1)

/var/folders/g9/k5r102q54f126nrlr692zy6c0000gn/T/ipykernel_37999/3012130192.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  unique_dicom['dbt_available'] = unique_dicom['dbt_available'].replace('DBT', 1)


In [15]:
unique_dicom

,PATIENT_STUDY_ID,ACCESSION_NUMBER,StudyDate,StudyTime,study,image_available,dbt_available
0,4330018595,60103700,2020-06-02,09:10:52,DIAG,1,NaN
1,4330018595,60690108,2020-06-02,08:33:36,SCREEN,1,NaN
2,4330018595,63737104,2019-08-19,02:11:20,DIAG,1,NaN
3,4330029102,64888584,2019-05-16,08:32:09,SCREEN,1,NaN
4,4330044371,61647696,2020-02-06,15:37:19,SCREEN,1,NaN
...,...,...,...,...,...,...,...
15135,4339943307,63434181,2019-06-17,17:07:31,DIAG,1,NaN
15136,4339943307,63570279,2019-05-23,18:36:05,SCREEN,1,NaN
15137,4339945522,62906727,2019-10-18,03:00:54,DIAG,1,NaN
15138,4339945522,450264459,2022-10-14,11:54:03,NaN,1,NaN


## B. Electric Health Record

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [16]:
file_path = "../Data/parameters of interest.xlsx"
file_name = pd.ExcelFile(file_path).sheet_names
file_name

['pathology',
 'pathology_findings',
 'patient_data_ie',
 'family_hx',
 'patient_demo',
 'vitals',
 'risk_factors',
 'enteredit_findings',
 'hormonal_mens']

---

# I. Construct <span style="color:blue;">**Control**</span> pool 
#### output file = "control_cohort", N=299

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

## 1. Merge with enteredit_findings 

In [23]:
fn

'enteredit_findings'

#### <span style="color:Red;">**[SKIP]**</span> Only 26 patients from the control group have corresponding data in DICOM, and these 26 patients are also found in the cancer dataset → Use <span style="color:blue;">**Cancer**</span> data for the following <span style="color:Red;">**[SKIP]**</span> 

##### Check if any control in dicom

In [24]:
file_path = os.path.join("../Data/Control/Cleaned", fn + ".xlsx")
BIRADS_control = pd.read_excel(file_path)

In [25]:
BIRADS_control = BIRADS_control[BIRADS_control['PATIENT_STUDY_ID'].isin(PIDs)]

In [27]:
BIRADS_control['PATIENT_STUDY_ID'].unique().size

26

##### Check if cancer/control overlap

In [28]:
file_path = os.path.join("../Data/Cancer/Cleaned", fn + ".xlsx")
BIRADS_cancer = pd.read_excel(file_path)

In [29]:
BIRADS_cancer = BIRADS_cancer[BIRADS_cancer['PATIENT_STUDY_ID'].isin(PIDs)]

In [30]:
set_control = set(BIRADS_control["PATIENT_STUDY_ID"].unique())
set_cancer = set(BIRADS_cancer["PATIENT_STUDY_ID"].unique())

set_duplicates = set_control & set_cancer
len(set_control), len(set_cancer), len(set_duplicates)

(26, 5509, 26)

#### <span style="color:Red;">**Continue from here**</span> 

In [31]:
file_path = os.path.join("../Data/Cancer/Cleaned", fn + ".xlsx")
BIRADS = pd.read_excel(file_path)

In [32]:
BIRADS = BIRADS[BIRADS['PATIENT_STUDY_ID'].isin(PIDs)]

In [33]:
set_dicom_PIDs = set(PIDs)
set_BIRADS_PIDs = set(BIRADS["PATIENT_STUDY_ID"].unique())

common_PIDs = set_dicom_PIDs & set_BIRADS_PIDs

In [34]:
len(common_PIDs)

5509

In [35]:
unique_dicom[unique_dicom["PATIENT_STUDY_ID"]==4330344296]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,StudyDate,StudyTime,study,image_available,dbt_available
66,4330344296,62861703,2019-12-20,02:44:43,SCREEN,1,NaN
67,4330344296,67469599,2020-12-31,08:39:44,SCREEN,1,NaN
68,4330344296,71644307,2017-07-31,02:23:29,SCREEN,1,NaN
69,4330344296,76610889,2018-10-31,16:35:19,SCREEN,1,NaN
70,4330344296,76695188,2018-11-08,15:23:00,DIAG,1,NaN


In [36]:
BIRADS[BIRADS["PATIENT_STUDY_ID"]==4330344296]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count
402,4330344296,71644307,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-07-31,2
403,4330344296,76610889,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2018-10-31,1
404,4330344296,76610889,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-10-31,1
405,4330344296,76695188,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-11-08,1
406,4330344296,62861703,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-12-20,2
407,4330344296,67469599,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-12-31,1
408,4330344296,67469599,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2020-12-31,1
409,4330344296,67156697,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2
410,4330344296,453737129,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-02-10,1


In [37]:
merge = pd.merge(
    BIRADS, 
    unique_dicom,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], 
    how='left'
)

In [38]:
merge[merge["PATIENT_STUDY_ID"]==4330344296]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,study,image_available,dbt_available
250,4330344296,71644307,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-07-31,2,2017-07-31,02:23:29,SCREEN,1.0,NaN
251,4330344296,76610889,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2018-10-31,1,2018-10-31,16:35:19,SCREEN,1.0,NaN
252,4330344296,76610889,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-10-31,1,2018-10-31,16:35:19,SCREEN,1.0,NaN
253,4330344296,76695188,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-11-08,1,2018-11-08,15:23:00,DIAG,1.0,NaN
254,4330344296,62861703,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-12-20,2,2019-12-20,02:44:43,SCREEN,1.0,NaN
255,4330344296,67469599,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2020-12-31,1,2020-12-31,08:39:44,SCREEN,1.0,NaN
256,4330344296,67469599,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2020-12-31,1,2020-12-31,08:39:44,SCREEN,1.0,NaN
257,4330344296,67156697,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2021-01-25,2,NaN,NaN,NaN,NaN,NaN
258,4330344296,453737129,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-02-10,1,NaN,NaN,NaN,NaN,NaN


In [39]:
merge_copy = merge[merge["PATIENT_STUDY_ID"].isin(PIDs)].copy()

In [40]:
merge_copy["PATIENT_STUDY_ID"].unique().size

5509

## 2. Merge with pathology

#### Store <span style="color:blue;">**control**</span> record only

In [41]:
fn = file_name[0]
fn

'pathology'

In [42]:
file_path = os.path.join("../Data/Cancer/Cleaned", fn + ".xlsx")
pathology = pd.read_excel(file_path)

#### Exclude patients having pathology records, N=5509→299

In [57]:
pids_path = pathology['PATIENT_STUDY_ID'].unique()

In [58]:
cohort_w_path = merge_copy[merge_copy["PATIENT_STUDY_ID"].isin(pids_path)].reset_index(drop=True)
cohort = merge_copy[~merge_copy["PATIENT_STUDY_ID"].isin(pids_path)].reset_index(drop=True)

In [59]:
print("cohort with pathology:", cohort_w_path["PATIENT_STUDY_ID"].unique().size)
print("cohort withthout pathology:", cohort["PATIENT_STUDY_ID"].unique().size)

cohort with pathology: 5210
cohort withthout pathology: 299


In [64]:
cohort = cohort.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER']).reset_index(drop=True)

In [65]:
output_file = "../Data/Control/control_cohort.xlsx"
cohort.to_excel(output_file, index=False)

## 3. Fill missing <span style="color:blue"> **Study**</span> (SCREEN, DIAG) with patient_data_ie

#### <span style="color:#FF6347;">**READ**</span> file

In [66]:
file_path = "../Data/Control/control_cohort.xlsx"
cohort = pd.read_excel(file_path)

In [67]:
file_path = "../Data/Cancer/patient_data_ie_tag.xlsx"
pdi = pd.read_excel(file_path)

#### Fill missing <span style="color:blue"> **Study**</span>  with patient_data_ie

In [68]:
pdi.columns

Index(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'ORG_NAME', 'EXAM_CODE',
       'EXAM_NAME', 'COMPLETED_DATE', 'ORDER_DATE', 'EXAM_TYPE',
       'FIRST_MAMMO_IND', 'LAST_ACTIVITY_DATE', 'INTERNAL_EXAM_ID',
       'PATIENT_ID', 'study'],
      dtype='object')

In [69]:
extract_cols = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'EXAM_TYPE', 'study']

In [71]:
cohort_copy = cohort.copy()
pdi_copy = pdi[extract_cols].copy()

In [72]:
cohort_cols = cohort_copy.columns
pdi_cols = pdi_copy.columns

In [73]:
merge = pd.merge(cohort_copy, pdi_copy, how="left", on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'])

In [76]:
merge[merge["PATIENT_STUDY_ID"]==4339688328]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,study_x,image_available,dbt_available,EXAM_TYPE,study_y
2714,4339688328,71023963,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2017-07-15,1,2017-07-15,18:14:14,SCREEN,1.0,NaN,Mammography,SCREEN
2715,4339688328,71023963,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2017-07-15,1,2017-07-15,18:14:14,SCREEN,1.0,NaN,Mammography,SCREEN
2716,4339688328,70541847,Heterogeneously dense (51% - 75%),NaN,4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2017-08-28,1,2017-08-28,09:06:30,DIAG,1.0,NaN,Digital Mammography,DIAG
2717,4339688328,455118390,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2021-07-09,1,NaN,NaN,NaN,NaN,NaN,Mammography,SCREEN
2718,4339688328,455118390,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2021-07-09,1,NaN,NaN,NaN,NaN,NaN,Mammography,SCREEN


In [77]:
# Create the final 'study' column by prioritizing cancer cohort (study info from DICOM), then patient_data_ie
merge['study'] = merge['study_x'].combine_first(merge['study_y'])
final = merge.drop(columns=['study_x', 'study_y'])

In [78]:
final[final["PATIENT_STUDY_ID"]==4339688328]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,image_available,dbt_available,EXAM_TYPE,study
2714,4339688328,71023963,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2017-07-15,1,2017-07-15,18:14:14,1.0,NaN,Mammography,SCREEN
2715,4339688328,71023963,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2017-07-15,1,2017-07-15,18:14:14,1.0,NaN,Mammography,SCREEN
2716,4339688328,70541847,Heterogeneously dense (51% - 75%),NaN,4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2017-08-28,1,2017-08-28,09:06:30,1.0,NaN,Digital Mammography,DIAG
2717,4339688328,455118390,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2021-07-09,1,NaN,NaN,NaN,NaN,Mammography,SCREEN
2718,4339688328,455118390,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2021-07-09,1,NaN,NaN,NaN,NaN,Mammography,SCREEN


In [80]:
output_file = os.path.join("../Data/Control/control_cohort.xlsx")
final.to_excel(output_file, index=False)

# II. Extract <span style="color:blue;">**Control**</span> record

## <span style="color:#FF6347;">**READ**</span> file (control_cohort)

In [3]:
output_file = os.path.join("../Data/", "Control", 'control_cohort' + ".xlsx")
cohort = pd.read_excel(output_file)

In [4]:
PIDs = cohort["PATIENT_STUDY_ID"].unique()

In [5]:
len(PIDs)

299

## 1. Locate <span style="color:blue;">**control**</span> <span style="color:#00BFFF;">**INDEX-1**</span> year, "index status" = <span style="color:#00BFFF;">**INDEX-1**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [6]:
cohort.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE'], inplace=True)
cohort['EXAM_COMPLETED_DATE'] = pd.to_datetime(cohort['EXAM_COMPLETED_DATE'])
cohort["index status"] = None

### 1a. Identify all <span style="color:#00BFFF;">**INDEX-1**</span> candidates 🎯

#### Find the record with "FINDING_CATEGORY"=="1 - Negative", "2 - Benign finding", and image_avaialbe, then marked the **earliest** exam date as <span style="color:#00BFFF;">**INDEX-1**</span>
#### **Criteria**:
#### 1. "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 2. "image_available" = 1
#### 3. "study" = "SCREEN"

In [7]:
idx_img = cohort["image_available"]==1
idx_dbt = cohort["dbt_available"]==1
idx_screen = cohort["study"]=="SCREEN"
idx_birads_0 = cohort["FINDING_CATEGORY"]=="0 - Need additional imaging evaluation"
idx_birads_1 = cohort["FINDING_CATEGORY"]=="1 - Negative"
idx_birads_2 = cohort["FINDING_CATEGORY"]=="2 - Benign finding"

In [8]:
cohort_candidate = cohort[idx_img & idx_screen & (idx_birads_1|idx_birads_2)]

In [9]:
pids_candidate = cohort[idx_img & idx_screen & (idx_birads_1|idx_birads_2)]['PATIENT_STUDY_ID'].unique()

In [10]:
len(pids_candidate)

207

### 1b. Mark the **earliest** date of <span style="color:#00BFFF;">**INDEX-1**</span> candidates as <span style="color:#00BFFF;">**INDEX-1**</span>✨

In [11]:
cohort_index_1 = cohort_candidate.drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first')
first_index_1_indices = cohort_index_1.index.tolist()

cohort.loc[first_index_1_indices, 'index status'] = 'INDEX-1'

In [12]:
cohort[cohort['PATIENT_STUDY_ID']==4330105691]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,image_available,dbt_available,EXAM_TYPE,study,index status
0,4330105691,68628042,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-11-23,1,NaN,NaN,NaN,NaN,Other,NaN,None
1,4330105691,68628042,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2020-11-23,1,NaN,NaN,NaN,NaN,Other,NaN,None
2,4330105691,66913098,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-06-14,1,NaN,NaN,NaN,NaN,Mammography,DIAG,None
3,4330105691,66913098,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2021-06-14,1,NaN,NaN,NaN,NaN,Mammography,DIAG,None
4,4330105691,66913944,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-06-14,1,NaN,NaN,NaN,NaN,Breast MRI,NaN,None
5,4330105691,66913944,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2021-06-14,1,NaN,NaN,NaN,NaN,Breast MRI,NaN,None
6,4330105691,451346204,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2022-07-18,1,2022-07-28,02:32:44,1.0,NaN,Mammography,SCREEN,None
7,4330105691,451346204,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2022-07-18,1,2022-07-28,02:32:44,1.0,NaN,Mammography,SCREEN,INDEX-1
8,4330105691,451346206,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-07-18,2,NaN,NaN,NaN,NaN,Breast MRI,NaN,None
9,4330105691,451963155,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-07-28,1,NaN,NaN,NaN,NaN,Mammography,DIAG,None


In [13]:
# Create a clean target DataFrame for the next step
target_df = cohort.loc[first_index_1_indices].copy().rename(
    columns={'EXAM_COMPLETED_DATE': 'target exam date'}
)

### 1c. Find the SCREEN/DIAGNOSIS exam that occurred within following 9 months of the <span style="color:#00BFFF;">**INDEX-1**</span> exam date, marked as <span style="color:#00BFFF;">**INDEX-1**</span>

In [14]:
# Locate Exams within 9 months of Index-1 Exam Date

tar_lookup = target_df.set_index('PATIENT_STUDY_ID')['target exam date']
cohort['target exam date'] = cohort['PATIENT_STUDY_ID'].map(tar_lookup)

# Calculate the time difference (INDEX-1 Exam Date - Exam Date)
cohort['exam to target diff for index-1'] = cohort['EXAM_COMPLETED_DATE'] - cohort['target exam date']

# Create a boolean mask
mask_preceding = cohort['exam to target diff for index-1'] >= pd.Timedelta(days=0)
mask_within_nine_mnths = cohort['exam to target diff for index-1'] < pd.Timedelta(days=365/12*9)
mask_index_1_cohort = mask_preceding & mask_within_nine_mnths

In [15]:
# Mark all rows matching the criteria as "index" (overwriting 'Other', retaining the original 'index' marks)
cohort.loc[mask_index_1_cohort, 'index status'] = 'INDEX-1'

print(f"✅ Complete. Marked {cohort[cohort['index status'] == 'INDEX-1'].shape[0]} rows as 'INDEX-1'.")

✅ Complete. Marked 401 rows as 'INDEX-1'.


In [16]:
cohort[cohort["PATIENT_STUDY_ID"]==4330105691]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,image_available,dbt_available,EXAM_TYPE,study,index status,target exam date,exam to target diff for index-1
0,4330105691,68628042,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-11-23,1,NaN,NaN,NaN,NaN,Other,NaN,None,2022-07-18,-602 days
1,4330105691,68628042,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2020-11-23,1,NaN,NaN,NaN,NaN,Other,NaN,None,2022-07-18,-602 days
2,4330105691,66913098,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-06-14,1,NaN,NaN,NaN,NaN,Mammography,DIAG,None,2022-07-18,-399 days
3,4330105691,66913098,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2021-06-14,1,NaN,NaN,NaN,NaN,Mammography,DIAG,None,2022-07-18,-399 days
4,4330105691,66913944,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-06-14,1,NaN,NaN,NaN,NaN,Breast MRI,NaN,None,2022-07-18,-399 days
5,4330105691,66913944,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2021-06-14,1,NaN,NaN,NaN,NaN,Breast MRI,NaN,None,2022-07-18,-399 days
6,4330105691,451346204,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2022-07-18,1,2022-07-28,02:32:44,1.0,NaN,Mammography,SCREEN,INDEX-1,2022-07-18,0 days
7,4330105691,451346204,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2022-07-18,1,2022-07-28,02:32:44,1.0,NaN,Mammography,SCREEN,INDEX-1,2022-07-18,0 days
8,4330105691,451346206,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-07-18,2,NaN,NaN,NaN,NaN,Breast MRI,NaN,INDEX-1,2022-07-18,0 days
9,4330105691,451963155,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-07-28,1,NaN,NaN,NaN,NaN,Mammography,DIAG,INDEX-1,2022-07-18,10 days


## 2. Locate <span style="color:blue;">**control**</span> <span style="color:#8A2BE2;">**INDEX**</span> year, "index status" = <span style="color:#8A2BE2;">**INDEX**</span> 

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

### 2a. Find the earliest exam date in <span style="color:#00BFFF;">**INDEX-1**</span> 


#### Already found in 1b

In [17]:
target_df 

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,target exam date,duplicate_count,StudyDate,StudyTime,image_available,dbt_available,EXAM_TYPE,study,index status
7,4330105691,451346204,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2022-07-18,1,2022-07-28,02:32:44,1.0,NaN,Mammography,SCREEN,INDEX-1
18,4330389368,76875097,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-11-28,2,2018-11-28,08:52:50,1.0,1.0,Mammography,SCREEN,INDEX-1
30,4330430143,67137254,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-02-01,1,2021-02-01,02:23:23,1.0,NaN,Mammography,SCREEN,INDEX-1
42,4330431872,76426245,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-10-12,2,2018-10-12,17:12:45,1.0,NaN,Mammography,SCREEN,INDEX-1
111,4333004308,72936269,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-03-29,2,2017-03-29,13:22:25,1.0,NaN,Mammography,SCREEN,INDEX-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2651,4335986882,61337298,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-01-24,2,2020-01-24,08:52:48,1.0,NaN,Mammography,SCREEN,INDEX-1
2660,4336406863,65273813,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-01-25,1,2019-01-25,18:36:02,1.0,NaN,Mammography,SCREEN,INDEX-1
2703,4337591637,65641557,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-02-20,2,2019-02-20,03:20:14,1.0,NaN,Mammography,SCREEN,INDEX-1
2709,4339426972,62629844,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-12-10,2,2019-12-10,17:07:52,1.0,NaN,Mammography,SCREEN,INDEX-1


In [18]:
cohort['target exam date']

0      2022-07-18
1      2022-07-18
2      2022-07-18
3      2022-07-18
4      2022-07-18
          ...    
2714   2017-07-15
2715   2017-07-15
2716   2017-07-15
2717   2017-07-15
2718   2017-07-15
Name: target exam date, Length: 2719, dtype: datetime64[ns]

### 2b. Find the following exam, <span style="color:blue;">**control**</span> <span style="color:#8A2BE2;">**INDEX**</span> year

#### Find the following exam that occurred at least 9 months after the earliest <span style="color:#00BFFF;">**INDEX-1**</span>  exam date but not beyond 18 months

In [19]:
cohort['exam to target diff for index'] = cohort['EXAM_COMPLETED_DATE'] - cohort['target exam date']

mask_after_nine_mnth = cohort['exam to target diff for index'] >= pd.Timedelta(days=365/12*9)
mask_within_eighteen_mnths = cohort['exam to target diff for index'] <= pd.Timedelta(days=365*1.5)
mask_index_cohort = mask_after_nine_mnth & mask_within_eighteen_mnths

In [20]:
# Mark all rows matching the criteria as "index" (overwriting 'Other', retaining the original 'index' marks)
cohort.loc[mask_index_cohort, 'index status'] = 'INDEX'

print(f"✅ Complete. Marked {cohort[cohort['index status'] == 'INDEX'].shape[0]} rows as 'INDEX'.")

✅ Complete. Marked 288 rows as 'INDEX'.


In [21]:
cohort["index status"].unique()

array([None, 'INDEX-1', 'INDEX'], dtype=object)

In [22]:
cohort_sort = cohort.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER']).reset_index(drop=True)

In [23]:
cohort_sort[cohort_sort["PATIENT_STUDY_ID"]==PIDs[5]]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,StudyDate,StudyTime,image_available,dbt_available,EXAM_TYPE,study,index status,target exam date,exam to target diff for index-1,exam to target diff for index
35,4330431872,72731854,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-03-30,1,NaN,NaN,NaN,NaN,Breast MRI,NaN,None,2018-10-12,-561 days,-561 days
36,4330431872,72731854,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2017-03-30,1,NaN,NaN,NaN,NaN,Breast MRI,NaN,None,2018-10-12,-561 days,-561 days
37,4330431872,70211453,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-10-13,1,2017-10-13,08:52:52,1.0,NaN,Breast MRI,NaN,None,2018-10-12,-364 days,-364 days
38,4330431872,70211453,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2017-10-13,1,2017-10-13,08:52:52,1.0,NaN,Breast MRI,NaN,None,2018-10-12,-364 days,-364 days
39,4330431872,70212626,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-10-13,1,2017-10-13,17:08:49,1.0,NaN,Digital Mammography,DIAG,None,2018-10-12,-364 days,-364 days
40,4330431872,70212626,Heterogeneously dense (51% - 75%),NaN,6 - Known biopsy proven malignancy,K-Appropriate action should be taken,2017-10-13,1,2017-10-13,17:08:49,1.0,NaN,Digital Mammography,DIAG,None,2018-10-12,-364 days,-364 days
41,4330431872,79589315,Heterogeneously dense (51% - 75%),NaN,99 - Post procedure mammograms for marker plac...,X-Follow-up post biopsy as directed by clinician,2017-10-31,1,NaN,NaN,NaN,NaN,Specimen Imaging,NaN,None,2018-10-12,-346 days,-346 days
42,4330431872,76426245,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2018-10-12,2,2018-10-12,17:12:45,1.0,NaN,Mammography,SCREEN,INDEX-1,2018-10-12,0 days,0 days
43,4330431872,65965175,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-02-01,1,NaN,NaN,NaN,NaN,Ultrasound,NaN,INDEX-1,2018-10-12,112 days,112 days
44,4330431872,62499697,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-10-18,2,2019-10-18,13:30:06,1.0,NaN,Mammography,SCREEN,INDEX,2018-10-12,371 days,371 days


In [24]:
output_file = os.path.join("../Data/", "Control", 'control_cohort_w_index' + ".xlsx")
cohort_sort.to_excel(output_file, index=False)

## 3. Extract <span style="color:blue;">**cleanest Control**</span> cohort only

### <span style="color:#FF6347;">**READ**</span> file (control_cohort_w_index)

In [3]:
output_file = os.path.join("../Data/", "Control", 'control_cohort_w_index' + ".xlsx")
cohort = pd.read_excel(output_file)

In [4]:
PIDs = cohort["PATIENT_STUDY_ID"].unique()

### **Criteria**:
#### 1. At <span style="color:#8A2BE2;">**INDEX**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding", and no pathology result available
##### No need to check as if pathology result is available or not. All patients in control_cohort already meet this criteria
#### 2. At <span style="color:#00BFFF;">**INDEX-1**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
##### ["1 - Negative", "2 - Benign finding"] = [>=90%, <=10%]
#### 3. At <span style="color:#00BFFF;">**INDEX-1**</span>, "image_available" = 1
#### 4. At <span style="color:#00BFFF;">**INDEX-1**</span>, "study" = "SCREEN"

In [5]:
idx_index = cohort["index status"]=="INDEX"
idx_index_1 = cohort["index status"]=="INDEX-1"
idx_img = cohort["image_available"]==1
idx_dbt = cohort["dbt_available"]==1
idx_screen = cohort["study"]=="SCREEN"
idx_birads_0 = cohort["FINDING_CATEGORY"]=="0 - Need additional imaging evaluation"
idx_birads_1 = cohort["FINDING_CATEGORY"]=="1 - Negative"
idx_birads_2 = cohort["FINDING_CATEGORY"]=="2 - Benign finding"

#### [N=162] Meet **criteria #1**

In [6]:
pids_criteria_1 = cohort[idx_index & (idx_birads_1 | idx_birads_2)]["PATIENT_STUDY_ID"].unique()

In [7]:
len(pids_criteria_1)

162

#### [N=64] Meet **criteria #1 <span style="color:red">**AND**</span> #3 <span style="color:red">**AND**</span> #4**

In [8]:
pids_criteria_1_3_4 = cohort[idx_img & idx_screen & idx_index & (idx_birads_1 | idx_birads_2)]["PATIENT_STUDY_ID"].unique()

In [9]:
len(pids_criteria_1_3_4)

64

#### [N=207] Meet **criteria #2** 

In [10]:
pids_criteria_2 = cohort[idx_index_1 & (idx_birads_1 | idx_birads_2)]["PATIENT_STUDY_ID"].unique()

In [11]:
len(pids_criteria_2)

207

#### [N=207] Meet **criteria #2 <span style="color:red">**AND**</span> #3 <span style="color:red">**AND**</span> #4**

In [12]:
pids_criteria_2_3_4 = cohort[idx_img & idx_screen & idx_index_1 & (idx_birads_1 | idx_birads_2)]["PATIENT_STUDY_ID"].unique()

In [13]:
len(pids_criteria_2_3_4)

207

#### [N=35] Meet **criteria #1 <span style="color:red">**AND**</span> #2 <span style="color:red">**AND**</span> #3 <span style="color:red">**AND**</span> #4** → Remove PID with other BIRADS score (not 1 or 2) at both <span style="color:#00BFFF;">**INDEX-1**</span> and <span style="color:#8A2BE2;">**INDEX**</span> → <span style="color:#FF6347;">**SAVE AS**</span> **clean_control_cohort**

##### A. [N=64] Meet **criteria #1 <span style="color:red">**AND**</span> #2 <span style="color:red">**AND**</span> #3 <span style="color:red">**AND**</span> #4**

In [14]:
pids_criteria_all = list(set(pids_criteria_1_3_4.tolist())&set(pids_criteria_2_3_4.tolist()))

In [15]:
len(pids_criteria_all)

64

##### Exclude PIDs with BIRADS that is not 1 or 2

In [16]:
findings_to_exclude = ["1 - Negative", "2 - Benign finding"]

##### B. PID with any BIRADS score other than 1 or 2 at <span style="color:#00BFFF;">**INDEX-1**</span> 

In [17]:
tmp_cohort_index_1 = cohort[idx_index_1]

In [18]:
filter_mask = ~tmp_cohort_index_1['FINDING_CATEGORY'].isin(findings_to_exclude)
pids_exclude_index_1 = tmp_cohort_index_1[filter_mask]['PATIENT_STUDY_ID'].tolist()

##### C. PID with any BIRADS score other than 1 or 2 at <span style="color:#8A2BE2;">**INDEX**</span>

In [19]:
tmp_cohort_index = cohort[idx_index]

In [20]:
filter_mask = ~tmp_cohort_index['FINDING_CATEGORY'].isin(findings_to_exclude)
pids_exclude_index = tmp_cohort_index[filter_mask]['PATIENT_STUDY_ID'].tolist()

##### $A - (B \cup C)$

In [21]:
set_A = set(pids_criteria_all)
set_B = set(pids_exclude_index_1)
set_C = set(pids_exclude_index)

In [22]:
set_B_or_C = set_B.union(set_C)
final_pids = list(set_A.difference(set_B_or_C))

In [23]:
len(final_pids)

35

#### <span style="color:#FF6347;">**SAVE**</span> **clean_control_cohort**

In [24]:
cohort_filtered = cohort[cohort['PATIENT_STUDY_ID'].isin(final_pids)].reset_index(drop=True)

In [25]:
cohort_filtered_sort = cohort_filtered.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER']).reset_index(drop=True)

In [26]:
output_file = os.path.join("../Data/", "Control", 'clean_control_cohort' + ".xlsx")
cohort_filtered_sort.to_excel(output_file, index=False)

## <span style="color:#FF6347;">**READ**</span> file

In [27]:
output_file = os.path.join("../Data/", "Control", 'clean_control_cohort' + ".xlsx")
cohort = pd.read_excel(output_file)

In [28]:
PIDs = cohort["PATIENT_STUDY_ID"].unique()

In [29]:
len(PIDs)

35

#### [N=35] Meet **criteria #1 <span style="color:red">**AND**</span> #2 <span style="color:red">**AND**</span> #3 <span style="color:red">**AND**</span> #4** → Remove PID with other BIRADS score (not 1 or 2) at both <span style="color:#00BFFF;">**INDEX-1**</span> and <span style="color:#8A2BE2;">**INDEX**</span> → <span style="color:#FF6347;">**SAVE AS**</span> **clean_control_cohort_dbt**

In [30]:
idx_dbt = cohort["dbt_available"]==1

In [31]:
pids_dbt = cohort.loc[idx_dbt, "PATIENT_STUDY_ID"].unique().tolist()

In [32]:
len(pids_dbt)

8

#### <span style="color:#FF6347;">**SAVE**</span> **clean_control_cohort_dbt**

In [33]:
cohort_filtered = cohort[cohort['PATIENT_STUDY_ID'].isin(pids_dbt)].reset_index(drop=True)

In [34]:
cohort_filtered_sort = cohort_filtered.sort_values(['PATIENT_STUDY_ID', 'EXAM_COMPLETED_DATE', 'ACCESSION_NUMBER']).reset_index(drop=True)

In [35]:
output_file = os.path.join("../Data/", "Control", 'clean_control_cohort_dbt' + ".xlsx")
cohort_filtered_sort.to_excel(output_file, index=False)